In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [2]:
from biked_commons.prediction import loaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
model_path = models_and_scalers_path("validity_model.pt")
scaler_path = models_and_scalers_path("validity_scaler.pt")
preprocessor = Preprocessor(scaler_path=scaler_path, preprocess_fn=None, device=device)
converter = framed.clip_to_framed_tensor_builder(ordered_columns.ORDERED_COLUMNS, framed.FRAMED_ORDERED_COLUMNS)
model = torch.load(model_path, weights_only=False).to(device)

data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)
data_tens = torch.tensor(data.values, dtype=dtype).to(device)

data_framed, _ = loaders.load_validity()
framed_tensor = torch.tensor(data_framed.values.astype(float), dtype=dtype).to(device)

regen_framed_tensor = converter(data_tens)
regen_framed_tensor = regen_framed_tensor.to(device, dtype=dtype)


preprocessed = preprocessor(framed_tensor)
predictions = model(preprocessed)
validity = predictions-0.5

In [3]:
regen_framed_tensor

tensor([[0.0000e+00, 7.6800e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         6.5933e-01],
        [0.0000e+00, 3.3000e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         5.7224e-01],
        [0.0000e+00, 7.6300e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         6.4709e-01],
        ...,
        [0.0000e+00, 7.4000e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         5.4775e-01],
        [0.0000e+00, 7.4000e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         5.4220e-01],
        [0.0000e+00, 7.7500e+02, 0.0000e+00,  ..., 1.0000e-03, 1.1000e-03,
         6.5499e-01]], device='cuda:0')

In [4]:
data_framed

,Material=Steel,Material=Aluminum,Material=Titanium,SSB_Include,CSB_Include,CS Length,BB Drop,Stack,SS E,ST Angle,...,CSB Offset,SS Z,SS Thickness,CS Thickness,TT Thickness,BB Thickness,HT Thickness,ST Thickness,DT Thickness,DT Length
9237,True,False,False,0.0,0.0,0.454210,4.974500e-02,0.56541,0.028226,74.602428,...,0.381567,0.010334,0.006401,0.001903,0.003043,0.003054,0.008850,0.002345,0.006206,0.660599
8922,True,False,False,1.0,1.0,0.428072,7.297700e-02,0.56664,0.034613,72.058553,...,0.299777,0.008548,0.006327,0.002624,0.002506,0.000540,0.002825,0.000872,0.003573,0.674965
11585,False,True,False,1.0,1.0,0.358010,-2.328000e-02,0.55498,0.175450,74.661000,...,0.349621,0.008206,0.003124,0.006949,0.000855,0.005617,0.001092,0.001054,0.004003,0.568519
6892,False,False,True,1.0,1.0,0.404602,7.005700e-02,0.56557,0.044059,74.011709,...,0.350116,0.008996,0.001073,0.001192,0.000751,0.000813,0.000962,0.000798,0.000956,0.658852
4783,False,True,False,1.0,1.0,0.297080,-1.282000e-02,0.56564,0.053940,66.369610,...,0.300662,0.009216,0.001014,0.001998,0.001276,0.000651,0.008124,0.008348,0.005268,0.415527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7997,True,False,False,0.0,0.0,0.425420,2.526700e-02,0.56560,0.045000,72.488558,...,0.300000,0.009000,0.003461,0.001703,0.003410,0.001102,0.004535,0.002055,0.009791,0.674373
13948,True,False,False,0.0,0.0,0.409670,5.098800e-02,0.56523,0.039210,73.530997,...,0.290824,0.008977,0.002399,0.002827,0.005921,0.002497,0.003042,0.006572,0.002998,0.665984
1307,True,False,False,0.0,0.0,0.362440,1.989520e-15,0.56560,0.045000,74.000000,...,0.300000,0.009000,0.002200,0.002957,0.001814,0.000863,0.001952,0.003190,0.002940,0.570119
8870,False,False,True,1.0,1.0,0.392032,6.675600e-02,0.53600,0.173262,75.240875,...,0.349996,0.008863,0.001042,0.000980,0.003368,0.003429,0.001838,0.003297,0.003799,0.645932


In [5]:
np.max(predictions.cpu().detach().numpy())

np.float32(1.0)

In [6]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [7]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [8]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [9]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [10]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


torch.Size([4510, 94])
tensor([-2.4708e+00,  3.1470e+00,  1.8289e+00,  8.1460e+00, -3.8564e+00,
         6.6942e-01, -6.7153e-01, -5.1108e+00, -2.5226e+00, -9.4910e-01,
        -1.7396e-01, -3.1535e-01, -5.8118e-01, -1.3241e-01,  2.2539e-01,
        -5.6002e-02,  2.3076e-02,  1.0031e+00,  1.2599e-02,  3.7831e+00,
        -3.8227e-03,  7.3053e-02,  3.7937e-02,  3.0863e-03, -1.1037e-01,
        -1.0603e-02, -7.1568e-01, -6.7448e-01, -1.3151e-02, -2.2444e+00,
        -1.8268e+00,  3.8245e-02, -2.4888e-01,  0.0000e+00,  5.0000e-01,
         1.2343e-07,  5.0000e-01, -1.8355e-08, -3.1875e-06,  7.5276e-08,
         5.8358e-02, -5.3158e-02,  1.3277e-07, -9.6857e-02,  1.5066e-06,
        -5.3626e-07, -3.8959e-06,  1.1844e-05,  7.8558e-07,  6.3653e-06,
         4.0495e-06,  1.0000e+00, -1.0000e+00, -1.0000e+00, -2.7596e-07,
        -4.1206e-07, -5.0346e-08,  4.2357e-09,  3.1954e-09,  3.5456e+00,
        -1.0900e-08, -1.0000e+00, -2.5855e-05, -3.9906e+00,  1.4498e+00,
        -3.0140e-05, -2.2830

In [11]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [12]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

Reference point does not include all objective names. Recomputing...
Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [13]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000e+00
Constraint Satisfaction Rate    8.596009e-01
Maximum Mean Discrepancy       -7.171762e-08
dtype: float64

In [14]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                  0.791411
Min Objective Score: Drag Force                                                                               27.732685
Min Objective Score: Knee Angle Error                                                                        197.293720
Min Objective Score: Hip Angle Error                                                                         805.295530
Min Objective Score: Arm Angle Error                                                                         842.159670
Min Objective Score: Cosine Similarity to Embedding                                                            0.286874
Min Objective Score: Mass                                                                                      5.618632
Min Objective Score: Planar Compliance                                                                      2071.537800
Min Objective Score: Transverse Complian